# 📝 Query Expansion - Cast a Wider Net!

## What is Query Expansion?

**Query expansion adds related terms, synonyms, and variations to your query to improve recall.**

### The Problem:
```python
Query: "car"
Docs with "automobile", "vehicle" → MISSED! ❌
```

### The Solution:
```python
Original: "car"
Expanded: "car automobile vehicle auto transportation"
Now finds ALL related docs! ✅
```

## Expansion Methods

### 1. **Synonym-Based** (WordNet)
- Uses linguistic databases
- Fast, deterministic
- Limited to known synonyms

### 2. **LLM-Based** ⭐ Most Powerful
- Generate contextual expansions
- Semantic understanding
- Domain-aware

### 3. **Embedding-Based**
- Find similar terms in embedding space
- Automatic, no manual rules
- Good for technical domains

### 4. **PRF (Pseudo Relevance Feedback)**
- Search, extract terms from top results
- Query again with expanded terms
- Self-improving

## 1. Synonym-Based Expansion (WordNet)

In [6]:
# Install nltk
# !pip install nltk
import warnings
warnings.filterwarnings("ignore") 

import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

# Download WordNet
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

def expand_with_wordnet(query, max_synonyms=3):
    """Expand query using WordNet synonyms"""
    tokens = word_tokenize(query.lower())
    expanded_terms = set(tokens)
    
    for token in tokens:
        # Get synsets (synonym sets)
        synsets = wordnet.synsets(token)
        
        # Add synonyms
        for syn in synsets[:max_synonyms]:
            for lemma in syn.lemmas()[:max_synonyms]:
                # Add if different from original
                synonym = lemma.name().replace('_', ' ').lower()
                if synonym != token:
                    expanded_terms.add(synonym)
    
    return ' '.join(expanded_terms)

# Test
queries = [
    "fast car",
    "happy person",
    "big house"
]

print("WordNet Query Expansion:\n")
print("="*80)

for query in queries:
    expanded = expand_with_wordnet(query)
    print(f"Original:  {query}")
    print(f"Expanded:  {expanded}\n")

print("💡 WordNet adds linguistic synonyms automatically!")

WordNet Query Expansion:

Original:  fast car
Expanded:  railway car auto fast fasting automobile gondola railcar car

Original:  happy person
Expanded:  happy glad felicitous someone person individual

Original:  big house
Expanded:  house firm business firm large big bad

💡 WordNet adds linguistic synonyms automatically!


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/anupamagaranisheshagiri/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2. LLM-Based Expansion (Best Quality)

In [7]:
# Simulated LLM expansion (use real LLM in production)
def expand_with_llm_simulated(query):
    """
    In production, replace with:
    - OpenAI GPT
    - Anthropic Claude  
    - Local LLM (Llama, Mistral)
    """
    # Simulated LLM responses
    llm_expansions = {
        "how to improve ml models": [
            "optimize machine learning algorithms",
            "enhance neural network performance",
            "boost AI model accuracy",
            "fine-tune deep learning systems",
            "improve training data quality"
        ],
        "best database for vectors": [
            "top vector database options",
            "efficient embedding storage solutions",
            "high-dimensional vector search systems",
            "similarity search databases"
        ]
    }
    
    # Find matching expansion
    for key in llm_expansions:
        if any(word in query.lower() for word in key.split()):
            return llm_expansions[key]
    
    return [query]

# Real LLM implementation template
def expand_with_llm_real(query):
    """
    Real implementation with OpenAI
    """
    prompt = f"""Generate 5 alternative phrasings for this query that capture the same intent:
    
Query: {query}

Alternative phrasings:
1."""
    
    # Call LLM (pseudo-code)
    # response = openai.Completion.create(prompt=prompt)
    # expansions = parse_expansions(response)
    
    return [query]  # Placeholder

# Test
query = "how to improve ml models"
expansions = expand_with_llm_simulated(query)

print(f"Query: '{query}'\n")
print("LLM-Generated Expansions:")
print("="*80)

for i, exp in enumerate(expansions, 1):
    print(f"{i}. {exp}")

print("\n🤖 LLM understands context and generates semantic variations!")

Query: 'how to improve ml models'

LLM-Generated Expansions:
1. optimize machine learning algorithms
2. enhance neural network performance
3. boost AI model accuracy
4. fine-tune deep learning systems
5. improve training data quality

🤖 LLM understands context and generates semantic variations!


## 3. Embedding-Based Expansion

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Domain vocabulary
vocab = [
    "machine learning", "deep learning", "neural networks", "algorithms",
    "artificial intelligence", "data science", "model training", "optimization",
    "accuracy", "performance", "fine-tuning", "hyperparameters",
    "embeddings", "transformers", "attention", "classification"
]

# Encode vocabulary
vocab_embeddings = model.encode(vocab)

def expand_with_embeddings(query, top_k=5):
    """Find similar terms from vocabulary using embeddings"""
    query_emb = model.encode(query)
    
    # Find most similar terms
    similarities = cosine_similarity([query_emb], vocab_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    similar_terms = [(vocab[i], similarities[i]) for i in top_indices]
    return similar_terms

# Test
query = "improving AI models"
similar_terms = expand_with_embeddings(query)

print(f"Query: '{query}'\n")
print("Embedding-Based Expansion:")
print("="*60)
print(f"{'Term':<30} {'Similarity'}")
print("="*60)

for term, sim in similar_terms:
    print(f"{term:<30} {sim:.4f}")

# Combine original + expansions
expanded_query = query + ' ' + ' '.join([term for term, _ in similar_terms[:3]])
print(f"\nExpanded Query: {expanded_query}")

print("\n✨ Embeddings find semantically similar terms!")

Query: 'improving AI models'

Embedding-Based Expansion:
Term                           Similarity
artificial intelligence        0.5398
model training                 0.5342
machine learning               0.4875
neural networks                0.4789
deep learning                  0.4679

Expanded Query: improving AI models artificial intelligence model training machine learning

✨ Embeddings find semantically similar terms!


## 4. Pseudo Relevance Feedback (PRF)

In [9]:
from collections import Counter
from nltk.corpus import stopwords
import string

nltk.download('stopwords', quiet=True)

# Sample documents
documents = [
    "Machine learning algorithms learn patterns from training data to make predictions.",
    "Deep neural networks use multiple layers for hierarchical feature learning.",
    "Model optimization improves accuracy through hyperparameter tuning.",
    "Fine-tuning pre-trained models adapts them to specific downstream tasks.",
    "Cross-validation helps evaluate model performance and prevent overfitting."
]

doc_embeddings = model.encode(documents)

def prf_expansion(query, documents, doc_embeddings, top_k=3, num_terms=5):
    """
    Pseudo Relevance Feedback:
    1. Search with original query
    2. Extract terms from top results
    3. Add terms to query
    4. Search again
    """
    # Initial search
    query_emb = model.encode(query)
    scores = cosine_similarity([query_emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # Extract terms from top documents
    stop_words = set(stopwords.words('english'))
    term_freq = Counter()
    
    for idx in top_indices:
        doc = documents[idx]
        # Tokenize and clean
        tokens = word_tokenize(doc.lower())
        tokens = [
            t for t in tokens 
            if t not in stop_words 
            and t not in string.punctuation
            and len(t) > 3
        ]
        term_freq.update(tokens)
    
    # Get top terms (exclude query terms)
    query_terms = set(query.lower().split())
    expansion_terms = [
        term for term, _ in term_freq.most_common(num_terms + 10)
        if term not in query_terms
    ][:num_terms]
    
    # Expanded query
    expanded = query + ' ' + ' '.join(expansion_terms)
    
    return expanded, top_indices, expansion_terms

# Test PRF
query = "model performance"
expanded_query, top_docs, terms = prf_expansion(query, documents, doc_embeddings)

print(f"Original Query: '{query}'\n")

print("Top Retrieved Docs (for feedback):")
for i, idx in enumerate(top_docs, 1):
    print(f"{i}. {documents[idx][:60]}...")

print(f"\nExtracted Terms: {terms}")
print(f"\nExpanded Query: '{expanded_query}'")

print("\n🔄 PRF uses initial results to improve the query!")

Original Query: 'model performance'

Top Retrieved Docs (for feedback):
1. Model optimization improves accuracy through hyperparameter ...
2. Cross-validation helps evaluate model performance and preven...
3. Fine-tuning pre-trained models adapts them to specific downs...

Extracted Terms: ['optimization', 'improves', 'accuracy', 'hyperparameter', 'tuning']

Expanded Query: 'model performance optimization improves accuracy hyperparameter tuning'

🔄 PRF uses initial results to improve the query!


## 5. Production Query Expander

In [10]:
class QueryExpander:
    def __init__(self, 
                 method='hybrid',
                 embedding_model='all-MiniLM-L6-v2',
                 vocab=None):
        """
        Production query expander
        
        Methods: 'wordnet', 'embeddings', 'prf', 'llm', 'hybrid'
        """
        self.method = method
        self.model = SentenceTransformer(embedding_model) if method in ['embeddings', 'prf', 'hybrid'] else None
        self.vocab = vocab
        self.vocab_embeddings = self.model.encode(vocab) if vocab and self.model else None
        
    def expand(self, query, top_k=3):
        """Expand query using selected method"""
        if self.method == 'wordnet':
            return self._expand_wordnet(query, top_k)
        elif self.method == 'embeddings':
            return self._expand_embeddings(query, top_k)
        elif self.method == 'hybrid':
            return self._expand_hybrid(query, top_k)
        else:
            return query
    
    def _expand_wordnet(self, query, max_syn=3):
        return expand_with_wordnet(query, max_syn)
    
    def _expand_embeddings(self, query, top_k):
        if not self.vocab_embeddings:
            return query
        
        query_emb = self.model.encode(query)
        sims = cosine_similarity([query_emb], self.vocab_embeddings)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        
        terms = [self.vocab[i] for i in top_idx]
        return query + ' ' + ' '.join(terms)
    
    def _expand_hybrid(self, query, top_k):
        """Combine multiple methods"""
        # WordNet synonyms
        wn_expanded = self._expand_wordnet(query, 2)
        
        # Embedding terms
        if self.vocab_embeddings:
            emb_expanded = self._expand_embeddings(query, top_k)
        else:
            emb_expanded = query
        
        # Combine unique terms
        all_terms = set(wn_expanded.split()) | set(emb_expanded.split())
        return ' '.join(all_terms)

# Test production expander
expander = QueryExpander(
    method='hybrid',
    vocab=vocab
)

test_queries = [
    "improve model accuracy",
    "fast training",
    "better predictions"
]

print("Production Query Expansion (Hybrid Method):\n")
print("="*80)

for query in test_queries:
    expanded = expander.expand(query, top_k=3)
    print(f"Original:  {query}")
    print(f"Expanded:  {expanded}\n")

print("✅ Hybrid expansion combines multiple techniques!")

Production Query Expansion (Hybrid Method):



ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

## 6. Measuring Expansion Impact

In [ ]:
# Compare search with and without expansion
from sklearn.metrics.pairwise import cosine_similarity

def search(query, doc_embeddings, documents, top_k=3):
    query_emb = model.encode(query)
    scores = cosine_similarity([query_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(documents[i], scores[i]) for i in top_idx]

# Test query
query = "optimize AI"

# Without expansion
results_basic = search(query, doc_embeddings, documents)

# With expansion
expanded_query = expander.expand(query, top_k=3)
results_expanded = search(expanded_query, doc_embeddings, documents)

print(f"Original Query: '{query}'")
print(f"Expanded Query: '{expanded_query}'\n")

print("Without Expansion:")
print("="*80)
for i, (doc, score) in enumerate(results_basic, 1):
    print(f"{i}. [{score:.4f}] {doc}")

print("\nWith Expansion:")
print("="*80)
for i, (doc, score) in enumerate(results_expanded, 1):
    print(f"{i}. [{score:.4f}] {doc}")

# Calculate improvement
avg_score_basic = np.mean([score for _, score in results_basic])
avg_score_expanded = np.mean([score for _, score in results_expanded])
improvement = ((avg_score_expanded - avg_score_basic) / avg_score_basic) * 100

print(f"\n📈 Average Score Improvement: {improvement:.1f}%")
print("💡 Expansion helps find more relevant results!")

## Key Takeaways 🎯

### ✅ Expansion Methods Comparison:

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| **WordNet** | Fast, deterministic | Limited coverage | General queries |
| **LLM** | Context-aware, powerful | Slow, costs $ | Complex queries |
| **Embeddings** | Domain-specific | Needs vocabulary | Technical domains |
| **PRF** | Self-improving | Two-pass search | When recall matters |
| **Hybrid** | Best quality | Most complex | Production |

### 🎯 When to Use Query Expansion:

**Use it when:**
- ✅ Recall is more important than precision
- ✅ Users use different terminology
- ✅ Domain has many synonyms
- ✅ Queries are often ambiguous

**Skip it when:**
- ❌ Precision is critical (may add noise)
- ❌ Queries are already detailed
- ❌ Latency is critical (<50ms)

### 💡 Best Practices:

```python
# 1. Start with simple methods
expander = QueryExpander(method='wordnet')

# 2. Build domain vocabulary for embeddings
vocab = extract_key_terms(your_documents)

# 3. Use LLM for complex queries only
if is_complex(query):
    expanded = llm_expand(query)

# 4. Combine methods for best results
expander = QueryExpander(method='hybrid')

# 5. Cache expansions
@lru_cache(maxsize=1000)
def cached_expand(query):
    return expander.expand(query)
```

### 📊 Expected Impact:

```
Recall Improvement:
- WordNet:     +10-15%
- Embeddings:  +15-20%
- LLM:         +20-30%
- PRF:         +25-35%
- Hybrid:      +30-40%

Precision Impact:
- May decrease 5-10% (more results = some noise)
- Use reranking to maintain precision
```

### 🚀 Production Template:

```python
class ProductionExpander:
    def __init__(self):
        # Multiple methods
        self.wordnet_expander = WordNetExpander()
        self.embedding_expander = EmbeddingExpander(vocab)
        self.llm_expander = LLMExpander()  # For complex only
        
    def expand(self, query):
        # Quick methods first
        expanded = self.wordnet_expander.expand(query)
        expanded = self.embedding_expander.expand(expanded)
        
        # LLM only if needed
        if self.is_complex(query):
            expanded = self.llm_expander.expand(expanded)
        
        return expanded
```

## Next Steps 🔜

Move to `02_Multi_Query_Generation.ipynb` for even more power!

Learn to generate multiple query variations automatically! 🎯